# Flows Basics [Step 1 - Event-driven Orchestration with CrewAI]

> **MLCourse - Agentic AI - CrewAI Flows and Orchestration**

CrewAI Flows are an event-driven orchestration layer that lets you compose
multi-step pipelines with typed state, conditional routing, and checkpointing.
A Flow is a Python class that inherits from `Flow` and decorates methods with
`@start()`, `@listen()`, and `@router()` to define execution order.

## What you will learn

1. The `Flow` base class and how to subclass it.
2. `@start()` marks the entry point -- the first method called on `kickoff()`.
3. `@listen(method_name)` chains a step to run after the listened method completes.
4. Typed state via Pydantic `BaseModel` -- `self.state` holds structured data.
5. Building a 3-step flow: research, write, review.

## Key takeaways

- Flows separate orchestration logic from agent logic.
- Each step receives the return value of the step it listens to.
- `self.state` is a typed Pydantic model, giving you autocomplete and validation.
- `flow.kickoff()` runs the entire pipeline synchronously.

In [ ]:
# --- Standard library imports -------------------------------------------------
import os                           # Environment variable access
from pathlib import Path            # OOP path handling

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv      # Load .env into os.environ

# Walk up from wherever this notebook was launched until we reach the track root
# folder "03_agentic_ai". This makes the notebook runnable from any subfolder,
# because the .env file with provider keys lives at that root (it is gitignored).
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")         # Load provider keys -- none needed for Ollama

# Guard IPython magic so the file stays valid pure Python outside Jupyter.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass                            # Not running inside IPython

print("Setup complete. Track root resolved to:", TRACK)

## 1 -- Verify Ollama availability

All CrewAI agents default to an LLM. We use `ChatOllama` with `llama3.2`
(local, free, no API key). The guard below confirms Ollama is reachable
before we build any agents.

In [ ]:
from langchain_ollama import ChatOllama  # Local LLM -- no API key needed

try:
    _test = ChatOllama(model="llama3.1:8b", temperature=0)
    _test.invoke("ping")
    LLM_AVAILABLE = True
    print("[GREEN] Ollama reachable -- full pipeline will run")
except Exception as exc:
    LLM_AVAILABLE = False
    print("[WARN] Ollama not reachable:", exc)
    print("Flow structure demonstrated without LLM calls")

## 2 -- Import CrewAI core classes

The key imports for flows:
- `Flow` -- the base class you subclass.
- `Agent` -- a role-playing agent with a goal, backstory, and optional tools.
- `Task` -- a unit of work assigned to an agent.
- `Crew` -- a group of agents and tasks executed together.
- `@start`, `@listen` -- decorators from `crewai.flow` that wire the DAG.

In [ ]:
from pydantic import BaseModel          # Typed state base class
from crewai import Flow, Agent, Task, Crew  # Core CrewAI classes
from crewai.flow import start, listen     # Flow wiring decorators

print("CrewAI imports successful")

## 3 -- Define typed flow state

Flows carry state via a Pydantic `BaseModel` subclass. Each field is typed
and validated. The state flows through every step -- methods read and write
`self.state.field_name`. We define three fields for our research-write-review
pipeline: the original topic, the research output, and the final draft.

In [ ]:
class ResearchWriteState(BaseModel):
    """Typed state for a 3-step research-write-review flow."""
    topic: str = ""          # Input topic provided at kickoff
    research: str = ""       # Output of the research step
    draft: str = ""          # Output of the writing step
    review: str = ""         # Output of the review step

print("State model defined:", ResearchWriteState.model_fields.keys())

## 4 -- Define agents

Each agent has a `role`, `goal`, and `backstory` (all plain strings that
guide the LLM's behavior). We create three agents: a researcher, a writer,
and a reviewer. All use `ChatOllama(model="llama3.2")`.

In [ ]:
llm = ChatOllama(model="llama3.1:8b", temperature=0) if LLM_AVAILABLE else None

researcher = Agent(
    role="Senior Research Analyst",
    goal="Conduct thorough research on the given topic and produce a structured summary",
    backstory="You are an expert researcher who synthesizes information clearly and concisely.",
    llm=llm,
    verbose=False,
)

writer = Agent(
    role="Technical Writer",
    goal="Transform research findings into a clear, well-structured draft",
    backstory="You are a skilled writer who makes complex topics accessible.",
    llm=llm,
    verbose=False,
)

reviewer = Agent(
    role="Editor and Reviewer",
    goal="Review the draft for accuracy, clarity, and completeness",
    backstory="You are a meticulous editor who catches errors and improves prose.",
    llm=llm,
    verbose=False,
)

print("Agents created:", [a.role for a in [researcher, writer, reviewer]])

## 5 -- Define tasks

Tasks are units of work. Each task has a `description` (what to do),
`expected_output` (what the result should look like), and an `agent`
(who does it). We use `{topic}` and `{research}` as placeholder variables
that CrewAI interpolates from the task's `context` at runtime.

In [ ]:
research_task = Task(
    description="Research the following topic thoroughly: {topic}",
    expected_output="A structured summary with key findings, facts, and insights.",
    agent=researcher,
)

write_task = Task(
    description="Write a clear draft based on this research: {research}",
    expected_output="A well-organized draft document of 200-400 words.",
    agent=writer,
)

review_task = Task(
    description="Review this draft for accuracy and clarity: {draft}",
    expected_output="A review with specific feedback and suggested improvements.",
    agent=reviewer,
)

print("Tasks created:", [t.description[:40] + "..." for t in [research_task, write_task, review_task]])

## 6 -- Build the Flow

Now we wire everything together in a `Flow` subclass. The key decorators:

- `@start()` -- marks the entry point. This method is called first when
  `flow.kickoff()` is invoked.
- `@listen(method_name)` -- chains a step to run after `method_name` completes.
  The return value of `method_name` is passed as the `result` argument.

Inside each step, we:
1. Create a `Crew` with the relevant agent and task(s).
2. Call `crew.kickoff()` to run the LLM pipeline.
3. Store the result in `self.state` for the next step.

In [ ]:
class ResearchWriteReviewFlow(Flow[ResearchWriteState]):
    """A 3-step flow: research -> write -> review.

    Each step creates a single-agent Crew, runs it, and stores the
    output in self.state so the next step can use it.
    """

    @start()
    def research(self):
        """Step 1: Research the topic stored in self.state.topic."""
        print(f"[Flow] Starting research on: {self.state.topic}")
        crew = Crew(
            agents=[researcher],
            tasks=[research_task],
            verbose=False,
        )
        # kickoff() accepts inputs that get interpolated into task descriptions.
        result = crew.kickoff(inputs={"topic": self.state.topic})
        self.state.research = str(result)
        print(f"[Flow] Research complete ({len(self.state.research)} chars)")
        return self.state.research

    @listen(research)
    def write(self, research_result):
        """Step 2: Write a draft based on the research output."""
        print(f"[Flow] Starting writing (research was {len(research_result)} chars)")
        crew = Crew(
            agents=[writer],
            tasks=[write_task],
            verbose=False,
        )
        result = crew.kickoff(inputs={"research": research_result})
        self.state.draft = str(result)
        print(f"[Flow] Writing complete ({len(self.state.draft)} chars)")
        return self.state.draft

    @listen(write)
    def review(self, draft_result):
        """Step 3: Review the draft and provide feedback."""
        print(f"[Flow] Starting review (draft is {len(draft_result)} chars)")
        crew = Crew(
            agents=[reviewer],
            tasks=[review_task],
            verbose=False,
        )
        result = crew.kickoff(inputs={"draft": draft_result})
        self.state.review = str(result)
        print(f"[Flow] Review complete ({len(self.state.review)} chars)")
        return self.state.review

print("Flow class defined:", ResearchWriteReviewFlow)

## 7 -- Run the flow

`flow.kickoff(inputs={...})` starts execution at the `@start()` method,
then follows the `@listen()` chain. The `inputs` dict seeds `self.state`
with the initial topic. After execution, `flow.state` contains the full
accumulated state.

In [ ]:
if LLM_AVAILABLE:
    flow = ResearchWriteReviewFlow()
    result = flow.kickoff(inputs={"topic": "The impact of retrieval-augmented generation on modern NLP"})

    print("\n" + "=" * 60)
    print("FLOW COMPLETE -- final state:")
    print("=" * 60)
    print(f"Topic:   {flow.state.topic}")
    print(f"Research: {flow.state.research[:200]}...")
    print(f"Draft:   {flow.state.draft[:200]}...")
    print(f"Review:  {flow.state.review[:200]}...")
else:
    print("Ollama not available -- demonstrating flow structure only")
    flow = ResearchWriteReviewFlow()
    # Show the flow structure without LLM calls
    print("Flow class methods:", [m for m in dir(flow) if not m.startswith('_')])

## 8 -- Inspect the flow structure

CrewAI Flows expose metadata about the DAG. We can inspect the flow
definition to see which methods are start points, which listen to what,
and the overall execution order.

In [ ]:
# Inspect the flow structure via the flow_definition property
flow_def = flow.flow_definition
print("Flow definition type:", type(flow_def))
print("Flow definition:", flow_def)

## 9 -- Understanding the execution model

Key points about how CrewAI Flows execute:

- **Synchronous kickoff**: `flow.kickoff()` runs all steps in order.
- **Async kickoff**: `flow.kickoff_async()` runs the flow asynchronously
  (returns a coroutine).
- **State persistence**: `@persist` decorator (covered in next module)
  checkpoints state to SQLite.
- **Conditional routing**: `@router()` (covered in advanced modules)
  allows branching based on step results.

The flow is essentially a directed acyclic graph (DAG) where edges are
defined by `@listen()` decorators. Each node is a method that creates
and runs a Crew.

In [ ]:
# Summary of what we built
print("=" * 60)
print("MODULE SUMMARY -- CrewAI Flows Basics")
print("=" * 60)
print()
print("Flow structure:")
print("  @start()  -> research()  -- entry point, seeds state.topic")
print("  @listen(research) -> write()  -- reads state.research, writes state.draft")
print("  @listen(write)    -> review() -- reads state.draft, writes state.review")
print()
print("Key classes:")
print("  Flow[StateType]  -- subclass with typed Pydantic state")
print("  Agent            -- role-playing LLM agent")
print("  Task             -- unit of work with description and expected output")
print("  Crew             -- group of agents + tasks executed together")
print()
print("Key decorators:")
print("  @start()         -- marks entry point method")
print("  @listen(method)  -- chains step to run after method completes")
print()
print("Key methods:")
print("  flow.kickoff(inputs={...})  -- run the flow synchronously")
print("  flow.state                  -- access typed state after execution")